# USP5 — Uji Per-Anggota Ensemble dan Uji Label Diacak

**Notebook ini dijalankan di kernel yang sama**, setelah semua sel di notebook sebelumnya (`USP5_Latih80_Uji53`) selesai. Ia memakai variabel dan fungsi dari sana (`X`, `y`, `groups`, `pipe`, `buat_fold_uji`, dan seterusnya). Cara termudah: salin sel-selnya ke bagian bawah notebook-mu.

## Untuk apa ujian ini?

Ensemble dosen berisi **tiga model** yang dirata-rata. Kamu sudah tahu hasilnya lumayan, tapi belum tahu **model mana yang sebenarnya membawa sinyal**. Ada dua ujian:

1. **Uji per-anggota (ablation).** Setiap anggota diuji sendirian, lalu dites "bagaimana kalau satu anggota dibuang". Ini menjawab: apakah ketiganya perlu? Apakah ada yang cuma "beban"? Apakah menggabungkan benar-benar menolong?
2. **Uji label diacak (Y-randomization).** Nilai pKd diacak, lalu model dilatih lagi. Kalau model tetap "bagus" pada label acak, hasil aslinya kebetulan. Kalau gagal total, hasil aslinya nyata. Ini ujian standar yang juga dilakukan di notebook dosen.

**Mengapa penting untuk skripsi?** Kamu bisa menjelaskan *kenapa* model bekerja, bukan hanya *seberapa bagus*, dan bisa membuktikan bahwa hasilnya bukan kebetulan.

## 1. Cek prasyarat

**Apa ini?** Memastikan semua variabel dan fungsi dari notebook sebelumnya masih ada di memori. Kalau ada yang hilang (misalnya kernel di-restart), sel ini berhenti dan memberi tahu apa yang harus dijalankan ulang.

In [ ]:
perlu = ['X', 'y', 'groups', 'is53', 'idx53', 'IDX_FPMA', 'IDX_DESCFP', 'pipe',
         'buat_fold_uji', 'indeks_latih', 'SEEDS', 'SKENARIO', 'OUT', 'hasil_seed',
         'r2_score', 'mean_squared_error', 'mean_absolute_error', 'spearmanr']
hilang = [n for n in perlu if n not in globals()]
assert not hilang, f'Jalankan dulu semua sel notebook sebelumnya. Yang hilang: {hilang}'
print('Prasyarat lengkap. Senyawa uji:', len(idx53), '| seed:', SEEDS)

## 2. Mendefinisikan anggota dan kombinasinya

**Apa ini?** Tiga anggota ensemble dosen **dibuat persis sama** seperti sebelumnya (pengaturannya tidak diubah), lalu disusun tujuh kombinasi untuk diuji:

- 3 kombinasi **satu anggota saja**,
- 3 kombinasi **tanpa satu anggota** (dua anggota dirata-rata),
- 1 kombinasi **ensemble penuh** (ketiganya), sebagai pembanding.

Cara membacanya nanti: kalau "tanpa X" turun jauh dibanding ensemble penuh, berarti X penting. Kalau hampir tidak berubah, X tidak banyak menyumbang.

In [ ]:
from sklearn.linear_model import BayesianRidge
from sklearn.svm import SVR
from sklearn.ensemble import GradientBoostingRegressor

def buat_anggota():
    # Sama persis dengan model_ensemble_lama() di notebook sebelumnya.
    return {
        'BayesianRidge':    pipe(IDX_FPMA,   BayesianRidge()),
        'SVR_linear':       pipe(IDX_FPMA,   SVR(kernel='linear', C=0.3, epsilon=0.05)),
        'GradientBoosting': pipe(IDX_DESCFP, GradientBoostingRegressor(n_estimators=400, learning_rate=0.05,
                                                                       max_depth=3, subsample=0.8, random_state=42)),
    }

NAMA_ANGGOTA = list(buat_anggota())
KOMBINASI = {
    'BR saja':            ['BayesianRidge'],
    'SVR saja':           ['SVR_linear'],
    'GB saja':            ['GradientBoosting'],
    'tanpa BR (SVR+GB)':  ['SVR_linear', 'GradientBoosting'],
    'tanpa SVR (BR+GB)':  ['BayesianRidge', 'GradientBoosting'],
    'tanpa GB (BR+SVR)':  ['BayesianRidge', 'SVR_linear'],
    'Ensemble penuh (3)': ['BayesianRidge', 'SVR_linear', 'GradientBoosting'],
}
print('Anggota  :', NAMA_ANGGOTA)
print('Kombinasi:', list(KOMBINASI))

## 3. Fungsi ujian per-anggota

**Apa ini?** Fungsi yang melatih setiap anggota **sekali per fold**, menyimpan prediksinya, lalu menghitung skor untuk ketujuh kombinasi dengan merata-ratakan prediksi anggota yang dipilih. Ini lebih cepat daripada melatih ulang tiap kombinasi.

Aturan pembagian data **sama persis** dengan ujian sebelumnya (senyawa uji hanya senyawa Mann, scaffold tidak bocor, skenario A dan B), jadi hasilnya bisa dibandingkan langsung.

In [ ]:
def evaluasi_anggota(skenario, kind, seed):
    pos = {int(i): p for p, i in enumerate(idx53)}
    pred_ang = {n: np.full(len(idx53), np.nan) for n in NAMA_ANGGOTA}
    tebak = np.full(len(idx53), np.nan)
    for uji in buat_fold_uji(kind, seed):
        latih = indeks_latih(skenario, uji, kind)
        assert set(latih).isdisjoint(set(uji)), 'Data uji bocor ke data latih'
        if kind == 'scaffold':
            assert set(groups[latih]).isdisjoint(set(groups[uji])), 'Scaffold bocor'
        p_uji = [pos[int(i)] for i in uji]
        for nama, est in buat_anggota().items():
            est.fit(X[latih], y[latih])
            pred_ang[nama][p_uji] = np.ravel(est.predict(X[uji]))
        tebak[p_uji] = y[latih].mean()
    yt = y[idx53]
    hasil = {}
    for kombi, daftar in KOMBINASI.items():
        p = np.mean([pred_ang[n] for n in daftar], axis=0)
        hasil[kombi] = dict(R2=r2_score(yt, p),
                            RMSE=float(np.sqrt(mean_squared_error(yt, p))),
                            MAE=mean_absolute_error(yt, p),
                            Spearman=float(spearmanr(yt, p).correlation))
    rmse_tebak = float(np.sqrt(mean_squared_error(yt, tebak)))
    return hasil, pred_ang, rmse_tebak

print('Fungsi siap')

## 4. Menjalankan ujian per-anggota

**Apa ini?** Menjalankan semua kombinasi untuk 2 protokol (scaffold dan acak), 2 skenario (A dan B), dan 5 seed. Hasil disimpan ke `hasil_anggota_per_seed.csv`. Perkiraan waktu: beberapa menit.

In [ ]:
PROTOKOL = ['scaffold', 'random']
baris, simpan_pred = [], {}
for kind in PROTOKOL:
    for sk in SKENARIO:
        for seed in SEEDS:
            hasil, pred_ang, rm_tebak = evaluasi_anggota(sk, kind, seed)
            simpan_pred[(kind, sk, seed)] = pred_ang
            for kombi, m in hasil.items():
                baris.append(dict(protokol=kind, skenario=sk, kombinasi=kombi, seed=seed,
                                  RMSE_tebak_rata2=rm_tebak, **m))
        print(f'selesai: {kind:8s} | {sk}')

hasil_ang = pd.DataFrame(baris)
hasil_ang.to_csv(OUT / 'hasil_anggota_per_seed.csv', index=False)
print('Tersimpan:', OUT / 'hasil_anggota_per_seed.csv')

## 5. Pemeriksaan: apakah cocok dengan hasil sebelumnya?

**Apa ini?** Kombinasi "Ensemble penuh (3)" seharusnya **sama persis** dengan `Ensemble_lama` di tabel sebelumnya (seed, fold, dan pengaturannya sama). Sel ini membandingkannya. Kalau cocok, berarti kode ujian ini benar. Kalau berbeda sedikit, biasanya karena versi library.

In [ ]:
lama = hasil_seed[hasil_seed['model'] == 'Ensemble_lama'][['protokol', 'skenario', 'seed', 'R2']]
baru = hasil_ang[hasil_ang['kombinasi'] == 'Ensemble penuh (3)'][['protokol', 'skenario', 'seed', 'R2']]
cek = lama.merge(baru, on=['protokol', 'skenario', 'seed'], suffixes=('_sebelumnya', '_sekarang'))
selisih = (cek['R2_sebelumnya'] - cek['R2_sekarang']).abs().max()
print('Baris yang dibandingkan :', len(cek))
print('Selisih R2 maksimum     :', f'{selisih:.2e}')
print('HASIL:', 'COCOK dengan hasil sebelumnya' if selisih < 1e-6 else 'ada selisih; periksa versi library atau seed')

## 6. Ringkasan per kombinasi

**Apa ini?** Tabel rata-rata (dan simpangan baku antar seed) untuk setiap kombinasi. Cara membaca:

- Bandingkan **"Ensemble penuh"** dengan **"X saja"**. Kalau ada anggota tunggal yang hampir sama bagusnya, penggabungan tidak menambah banyak.
- Bandingkan **"Ensemble penuh"** dengan **"tanpa X"**. Penurunan besar berarti X membawa sinyal. Tidak berubah berarti X tidak berpengaruh. Kalau skornya naik saat X dibuang, X justru mengganggu.
- Selisih yang lebih kecil dari `std` belum bisa dipercaya.

In [ ]:
urut = pd.CategoricalDtype(list(KOMBINASI), ordered=True)
hasil_ang['kombinasi'] = hasil_ang['kombinasi'].astype(urut)
ring = (hasil_ang.groupby(['protokol', 'skenario', 'kombinasi'], observed=True)[['R2', 'RMSE', 'Spearman']]
        .agg(['mean', 'std']).round(3))
ring.to_csv(OUT / 'ringkasan_anggota.csv')

print('=== PROTOKOL SCAFFOLD (utama) ===')
display(ring.loc['scaffold'])
print('=== PROTOKOL ACAK (referensi) ===')
display(ring.loc['random'])

## 7. Apakah kesalahan ketiga anggota saling berbeda?

**Apa ini?** Merata-ratakan model hanya menolong kalau **kesalahannya berbeda-beda**. Sel ini menghitung korelasi kesalahan (residual) antar anggota, memakai satu seed dan protokol scaffold skenario A.

- Korelasi **rendah** (misalnya di bawah 0,7): kesalahan berbeda, dan merata-ratakan menolong.
- Korelasi **sangat tinggi** (mendekati 1): anggota-anggotanya kembar, dan merata-ratakan hampir tidak ada gunanya.

In [ ]:
sk0, seed0 = SKENARIO[0], SEEDS[0]
pa = simpan_pred[('scaffold', sk0, seed0)]
resid = pd.DataFrame({n: y[idx53] - pa[n] for n in NAMA_ANGGOTA})
print(f'Korelasi residual antar anggota (scaffold, {sk0}, seed {seed0}):')
display(resid.corr().round(3))

## 8. Grafik per-anggota

**Apa ini?** Batang R² dan Spearman untuk tiap kombinasi pada protokol scaffold, dipisah antara skenario A dan B. Garis kecil di atas batang adalah simpangan baku antar seed.

In [ ]:
sc = hasil_ang[hasil_ang['protokol'] == 'scaffold']
agg = sc.groupby(['kombinasi', 'skenario'], observed=True)[['R2', 'Spearman']].agg(['mean', 'std'])
nama_k = list(KOMBINASI)
lebar = 0.38
fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
for a, metrik in zip(ax, ['R2', 'Spearman']):
    for j, sk in enumerate(SKENARIO):
        m = [agg.loc[(k, sk), (metrik, 'mean')] for k in nama_k]
        s = [agg.loc[(k, sk), (metrik, 'std')] for k in nama_k]
        a.bar(np.arange(len(nama_k)) + (j - 0.5) * lebar, m, lebar, yerr=s, capsize=3, label=sk.replace('_', ' '))
    a.axhline(0, color='k', lw=0.8)
    a.set_xticks(np.arange(len(nama_k)))
    a.set_xticklabels(nama_k, rotation=30, ha='right', fontsize=8)
    a.set_title(f'{metrik} (scaffold CV, rata-rata seed)')
    a.legend(fontsize=7)
plt.tight_layout()
plt.savefig(OUT / 'grafik_anggota_scaffold.png', dpi=200, bbox_inches='tight')
plt.show()

## 9. Uji label diacak (Y-randomization)

**Apa ini?** Nilai pKd diacak antar senyawa (struktur tetap, jawabannya dikocok), lalu **ensemble penuh dilatih dan diuji dengan aturan yang sama**. Ini diulang 20 kali.

- Pada label acak, model **harus gagal**: R² sekitar 0 atau negatif.
- Kalau R² asli jauh di atas semua hasil acak, berarti model menangkap hubungan struktur dan aktivitas yang nyata, bukan kebetulan.

Hanya satu seed dan satu skenario (scaffold, skenario A) yang dipakai supaya tidak terlalu lama. Nilai `y` asli otomatis dikembalikan di akhir sel (dan dicek).

In [ ]:
N_PERM = 20
kind_y, sk_y, seed_y = 'scaffold', SKENARIO[0], SEEDS[0]

# R2 asli untuk pembanding (dari hasil yang sama persis)
r2_asli = float(hasil_ang[(hasil_ang.protokol == kind_y) & (hasil_ang.skenario == sk_y) &
                          (hasil_ang.seed == seed_y) & (hasil_ang.kombinasi == 'Ensemble penuh (3)')]['R2'].iloc[0])

y_asli = y.copy()
rng = np.random.RandomState(0)
r2_acak = []
try:
    for i in range(N_PERM):
        y[:] = y_asli[rng.permutation(len(y_asli))]      # kocok jawaban (in-place)
        h, _, _ = evaluasi_anggota(sk_y, kind_y, seed_y)
        r2_acak.append(h['Ensemble penuh (3)']['R2'])
finally:
    y[:] = y_asli                                         # kembalikan y asli, apa pun yang terjadi
assert np.allclose(y, y_asli), 'y asli gagal dikembalikan!'

r2_acak = np.array(r2_acak)
p_empiris = (1 + (r2_acak >= r2_asli).sum()) / (1 + N_PERM)
print(f'R2 label asli   : {r2_asli:.3f}')
print(f'R2 label acak   : rata-rata {r2_acak.mean():.3f} | maks {r2_acak.max():.3f} | min {r2_acak.min():.3f}')
print(f'Nilai p empiris : {p_empiris:.3f}  (terkecil yang mungkin dengan {N_PERM} pengacakan = {1/(1+N_PERM):.3f})')
print('KESIMPULAN:', 'model berada di atas semua hasil label acak (sinyal terdeteksi)' if r2_asli > r2_acak.max()
      else 'model TIDAK jelas di atas label acak, hati-hati menafsirkan hasil')

plt.figure(figsize=(6, 3.6))
plt.hist(r2_acak, bins=12, color='#999999', edgecolor='white', label='label diacak')
plt.axvline(r2_asli, color='#C44E52', lw=2.5, label=f'label asli ({r2_asli:.3f})')
plt.axvline(0, color='k', lw=1, ls=':')
plt.xlabel('R2 out-of-fold'); plt.ylabel('frekuensi'); plt.title('Uji label diacak (scaffold, skenario A)')
plt.legend(fontsize=8); plt.tight_layout()
plt.savefig(OUT / 'grafik_y_randomization.png', dpi=200, bbox_inches='tight')
plt.show()

## 10. Cara melaporkan hasil ini dengan jujur

1. **Ini analisis eksplorasi.** Kalau kamu memilih satu anggota yang paling bagus berdasarkan tabel ini, itu sudah termasuk "menyetel pada data yang sama". Laporkan sebagai temuan eksplorasi, dan tetap jadikan **ensemble penuh** hasil utama.
2. **Laporkan semua tujuh kombinasi**, termasuk yang jelek.
3. **Kalimat yang aman** (sesuaikan dengan hasil aslimu): "Analisis ablasi menunjukkan bahwa [anggota X] menyumbang sebagian besar sinyal prediktif, sedangkan [anggota Y] tidak memperbaiki hasil secara bermakna. Uji label acak (20 pengacakan) menunjukkan R² jauh di bawah hasil asli, sehingga hasil tidak disebabkan korelasi kebetulan."
4. **Ingat batasannya:** hanya 53 senyawa uji dari satu seri kimia, dan simpangan baku antar seed terlalu optimis karena datanya sama.